# WP27 — Formal Invariant Verification
## Layer 10: State-Dependent Safety Invariants as Hoare Preconditions

---

This notebook demonstrates **WP27: Formal Invariant Verification**, which adds a
formal safety checker between the WP17–26 stack and the WP28 cross-domain transfer
layer. Every proposed synthesis action is verified against declared *safety invariants*
expressed as first-order constraints over the synthesis state.

### The Gap WP27 Closes

The `GodelianSafetyGovernor` used by earlier WPs classifies proposals by *type* only.
It has no knowledge of the *current state* of the MetaLearner. WP27 adds **state-dependent**
invariants — safety properties that depend on the current `SynthesisStateSnapshot`:

| Invariant | Description |
|-----------|-------------|
| `PROB_FLOOR` | Every strategy probability ≥ p_floor (default 0.02) |
| `ENTROPY_FLOOR` | Shannon entropy of probs ≥ H_min (default 0.5 nats) |
| `UPWARD_RATE_BOUNDS` | upward_rate ∈ [0.0, 1.0] after BOOST_UPWARD |
| `ACC_DROP_GUARD` | Accuracy has not dropped > delta in one generation |

### Hoare Triple Interpretation
> *{P} C {Q}: if the spec returns False, command C must not execute.*
> — C.A.R. Hoare (1969)

Runtime: **~5 min** (no GPU required)

In [ ]:
# ── 0. Environment setup ────────────────────────────────────────────────────
import sys, os, importlib

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('Prometheus_v0_PoC'):
        os.system('git clone https://github.com/pmcray/Prometheus_v0_PoC.git')
    os.chdir('Prometheus_v0_PoC')
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
    importlib.invalidate_caches()
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    importlib.invalidate_caches()
    print(f'Local mode — repo root: {repo_root}')

import warnings; warnings.filterwarnings('ignore')
import time, random, json
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from prometheus.wp27_formal_invariants import (
    InvariantCRLS, InvariantGuard, InvariantRegistry,
    InvariantSpec, InvariantRecord, InvariantCheckResult,
    verify_wp27_exit_criteria,
)
from prometheus.wp22_bandit_exploration import BanditMode
from prometheus.wp17_crls_synthesis import SynthesisAction
from prometheus.wp20_temporal_planner import SynthesisStateSnapshot
from prometheus.environments.go import GoBoard

import prometheus
print(f'Prometheus version: {prometheus.__version__}')
print('WP27 imports OK.')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams.update({'figure.figsize': (14, 7), 'font.size': 11})
SEED = 42; random.seed(SEED); np.random.seed(SEED)

In [ ]:
# ── 1. Configuration ────────────────────────────────────────────────────────
QUICK_MODE      = True
N_GENERATIONS   = 10 if QUICK_MODE else 30
PUZZLES_PER_GEN = 30 if QUICK_MODE else 80
BOARD_SIZE      = 9

REGIME_SEQUENCE = (
    ['ATARI'] * 4 + ['TERRITORY'] * 3 + ['MIXED'] * (N_GENERATIONS - 7)
)[:N_GENERATIONS]

print(f'Mode: {"QUICK" if QUICK_MODE else "FULL"}')
print(f'Generations: {N_GENERATIONS} | Regimes: {REGIME_SEQUENCE}')

In [ ]:
# ── 2. Puzzle factory ────────────────────────────────────────────────────────
def make_atari_puzzle(board_size, rng):
    board = GoBoard(size=board_size)
    cx = board_size // 2
    stones = [(cx, cx), (cx, cx+1), (cx+1, cx)]
    for r, c in stones:
        if board.is_on_board(r, c) and board.board[r, c] == GoBoard.EMPTY:
            board.board[r, c] = GoBoard.BLACK
    all_libs = set()
    for r, c in stones:
        for nr, nc in board.get_neighbors(r, c):
            if board.board[nr, nc] == GoBoard.EMPTY:
                all_libs.add((nr, nc))
    libs = list(all_libs); rng.shuffle(libs)
    for r, c in libs[:-1]: board.board[r, c] = GoBoard.WHITE
    board.current_player = GoBoard.WHITE
    return board, GoBoard.WHITE, libs[-1]

def make_territory_puzzle(board_size, rng):
    board = GoBoard(size=board_size)
    for r, c in [(0, 0), (0, board_size-1), (board_size-1, 0)]:
        board.board[r, c] = GoBoard.WHITE
    board.current_player = GoBoard.BLACK
    return board, GoBoard.BLACK, (board_size-1, board_size-1)

PUZZLE_FACTORIES = {'ATARI': make_atari_puzzle, 'TERRITORY': make_territory_puzzle}

def generate_puzzles(regime, n, board_size, seed):
    rng = np.random.default_rng(seed)
    if regime == 'MIXED':
        return [(PUZZLE_FACTORIES['ATARI'] if i % 2 == 0 else PUZZLE_FACTORIES['TERRITORY'])(board_size, rng) for i in range(n)]
    return [PUZZLE_FACTORIES[regime](board_size, rng) for _ in range(n)]

print('Puzzle factory OK.')

---
## Section 1 — InvariantRegistry: Demonstrating Individual Invariant Checks

Before running the full stack, we demonstrate the invariant system in isolation.
Each `InvariantSpec` is a predicate `(state, action) → bool` that must hold for
the action to be considered safe.

In [ ]:
# ── 3. Demonstrate invariant checking in isolation ───────────────────────────
# Build a synthetic SynthesisStateSnapshot to test invariants
def make_snapshot(accuracy=0.5, upward_rate=0.2, generation=5, probs=None):
    strategies = ['ATARI', 'LADDER', 'TERRITORY', 'MIXED']
    if probs is None:
        probs = {s: 0.25 for s in strategies}  # uniform
    return SynthesisStateSnapshot(
        generation=generation,
        accuracy=accuracy,
        upward_rate=upward_rate,
        probs=probs,
    )

# Test cases: (description, snapshot, action, should_pass)
registry = InvariantRegistry()  # uses default built-in invariants

test_cases = [
    (
        'Healthy state + PROMOTE_BEST',
        make_snapshot(accuracy=0.7, upward_rate=0.2,
                      probs={'ATARI': 0.4, 'LADDER': 0.3, 'TERRITORY': 0.2, 'MIXED': 0.1}),
        SynthesisAction.PROMOTE_BEST,
    ),
    (
        'Near-zero probs + DEMOTE_WORST (prob floor risk)',
        make_snapshot(probs={'ATARI': 0.97, 'LADDER': 0.01, 'TERRITORY': 0.01, 'MIXED': 0.01}),
        SynthesisAction.DEMOTE_WORST,
    ),
    (
        'Very high upward_rate + BOOST_UPWARD (rate bounds risk)',
        make_snapshot(upward_rate=0.98),
        SynthesisAction.BOOST_UPWARD,
    ),
    (
        'Uniform probs + RESET_UNIFORM (entropy floor: safe)',
        make_snapshot(probs={'ATARI': 0.25, 'LADDER': 0.25, 'TERRITORY': 0.25, 'MIXED': 0.25}),
        SynthesisAction.RESET_UNIFORM,
    ),
]

print('InvariantRegistry check results:')
print('=' * 65)
for desc, snapshot, action in test_cases:
    result = registry.check(snapshot, action)
    status = 'ALL PASS' if result.all_passed else f'VIOLATED: {[v for v in result.violated_specs]}'
    print(f'  [{"SAFE" if result.all_passed else "BLOCKED"}] {desc}')
    print(f'           → {status}')
    print()

---
## Section 2 — Full Stack: InvariantCRLS (10 Layers)

In [ ]:
# ── 4. Instantiate InvariantCRLS ──────────────────────────────────────────────
STRATEGIES = ['ATARI', 'LADDER', 'TERRITORY', 'MIXED']

stack = InvariantCRLS(
    strategies  = STRATEGIES,
    bandit_mode = BanditMode.UCB1,
)

print('InvariantCRLS (10 layers) instantiated.')
print('  Layers: WP17→WP19→WP20→WP21→WP22→WP23→WP24→WP25→WP26→WP27')

accuracies, invariant_records, block_counts = [], [], []
violated_names_all = []

In [ ]:
# ── 5. Main experiment loop ──────────────────────────────────────────────────
print('=' * 75)
print(f'  Gen  Regime        Acc     InvariantResult   Violated')
print('=' * 75)

for gen in range(N_GENERATIONS):
    regime = REGIME_SEQUENCE[gen]
    puzzles = generate_puzzles(regime, PUZZLES_PER_GEN, BOARD_SIZE, seed=gen * 137 + 1)

    acc = stack.run_generation(puzzles)
    ten_tuple = stack.end_of_generation()
    # 10-tuple: WP26 9-tuple + InvariantRecord
    inv_rec = ten_tuple[-1]   # InvariantRecord is the last element

    accuracies.append(acc)
    invariant_records.append(inv_rec)

    blocked    = not inv_rec.all_passed if inv_rec else False
    violated   = inv_rec.violated_specs if inv_rec else []
    block_counts.append(1 if blocked else 0)
    if violated:
        violated_names_all.extend(violated)

    result_str = 'BLOCKED' if blocked else 'PASSED'
    viol_str   = ', '.join(violated) if violated else '---'
    print(f'  {gen:3d}  {regime:<12}  {acc:.3f}  {result_str:<16}  {viol_str}')

print('=' * 75)
print(f'Mean accuracy: {np.mean(accuracies):.3f}')
print(f'Total blocks:  {sum(block_counts)} / {N_GENERATIONS}')

In [ ]:
# ── 6. Visualisation ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
gens = list(range(N_GENERATIONS))

regime_colors = {'ATARI': '#fff3cd', 'TERRITORY': '#d1ecf1', 'MIXED': '#e2d9f3'}
def add_bands(ax):
    prev, start = None, 0
    for g, r in enumerate(REGIME_SEQUENCE):
        if r != prev:
            if prev: ax.axvspan(start-.5, g-.5, alpha=.25, color=regime_colors.get(prev,'#eee'))
            start, prev = g, r
    ax.axvspan(start-.5, N_GENERATIONS-.5, alpha=.25, color=regime_colors.get(prev,'#eee'))

# A: Accuracy with block events
ax = axes[0, 0]
add_bands(ax)
ax.plot(gens, accuracies, 'g-o', linewidth=2.5, markersize=6)
for g, blocked in enumerate(block_counts):
    if blocked:
        ax.axvline(g, color='red', linewidth=2, linestyle=':', alpha=0.8)
        ax.text(g, 0.97, 'BLOCK', ha='center', va='top', fontsize=7, color='darkred', fontweight='bold')
ax.set_ylabel('Accuracy'); ax.set_title('Accuracy + WP27 Invariant Blocks\n(red = action blocked)', fontweight='bold')
ax.set_ylim(0, 1.05)

# B: Per-generation invariant result
ax2 = axes[0, 1]
colors_bar = ['#f44336' if b else '#4CAF50' for b in block_counts]
ax2.bar(gens, [1]*N_GENERATIONS, color=colors_bar, alpha=0.85, edgecolor='black')
passed_patch = mpatches.Patch(color='#4CAF50', label='All invariants passed')
blocked_patch = mpatches.Patch(color='#f44336', label='Invariant violated → blocked')
ax2.legend(handles=[passed_patch, blocked_patch], fontsize=9)
ax2.set_ylabel('Invariant result'); ax2.set_yticks([])
ax2.set_title('Per-Generation Invariant Check Result\n(WP27 safety gate)', fontweight='bold')

import matplotlib.patches as mpatches

# C: Which invariants were violated
ax3 = axes[1, 0]
if violated_names_all:
    vcounts = Counter(violated_names_all)
    ax3.bar(list(vcounts.keys()), list(vcounts.values()), color='#f44336', alpha=0.8, edgecolor='black')
    ax3.set_xlabel('Invariant name'); ax3.set_ylabel('Times violated')
else:
    ax3.text(0.5, 0.5, 'No invariant violations\nobserved in this run',
             ha='center', va='center', fontsize=13, transform=ax3.transAxes,
             bbox=dict(boxstyle='round', fc='#d4edda', alpha=0.9))
ax3.set_title('Invariant Violation Frequency\n(safety-critical events)', fontweight='bold')

# D: Summary text
ax4 = axes[1, 1]
ax4.axis('off')
report = stack.get_full_report()
inv_summary = report.get('invariant_summary', {})
summary_text = (
    'WP27 Formal Invariants — Summary\n'
    '════════════════════════════════\n\n'
    f'  Total checks:    {inv_summary.get("total_checks", N_GENERATIONS)}\n'
    f'  Total blocks:    {sum(block_counts)}\n'
    f'  Block rate:      {sum(block_counts)/N_GENERATIONS:.1%}\n\n'
    f'  Invariants enforced:\n'
    '    PROB_FLOOR (p ≥ 0.02)\n'
    '    ENTROPY_FLOOR (H ≥ 0.5)\n'
    '    UPWARD_RATE_BOUNDS\n'
    '    ACC_DROP_GUARD\n\n'
    'Hoare (1969):\n'
    '  {P} C {Q} — if precondition P\n'
    '  fails, command C is blocked.\n\n'
    'Every blocked action replaced\n'
    'by safe fallback: RESET_UNIFORM'
)
ax4.text(0.05, 0.95, summary_text, transform=ax4.transAxes,
         fontsize=9.5, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

fig.suptitle(
    'WP27: Formal Invariant Verification — Prometheus v0\n'
    'Layer 10: State-dependent safety invariants as Hoare preconditions',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('wp27_formal_invariants.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to wp27_formal_invariants.png')

In [ ]:
# ── 7. Verify WP27 exit criteria ─────────────────────────────────────────────
results = verify_wp27_exit_criteria(stack)
print('WP27 Exit Criteria Verification')
print('=' * 50)
all_pass = True
for criterion, passed in results.items():
    status = '✓ PASS' if passed else '✗ FAIL'
    print(f'  {status}  {criterion}')
    if not passed: all_pass = False
print()
if all_pass:
    print('All WP27 exit criteria satisfied.')
    print('The invariant guard is verifying synthesis actions against formal')
    print('safety constraints before they are applied.')
else:
    print('Some criteria not yet met — run more generations.')

---
## Conclusions

**Formal invariant verification** (Hoare 1969, Lamport 1977) transforms safety
from an implicit hope into an explicit, auditable contract. Each `InvariantSpec`
is a Hoare precondition: if the synthesis state violates the precondition for a
proposed action, the action is blocked and replaced by a safe fallback.

This is a concrete realisation of Good's requirement that an ultraintelligent machine
must be able to *verify* that its self-modification proposals are safe before applying them.

### References
- Hoare, C.A.R. (1969). An axiomatic basis for computer programming. *CACM*, 12(10), 576–580.
- Lamport, L. (1977). Proving the correctness of multiprocess programs. *IEEE TSE*, 3(2), 125–143.
- Clarke, E.M., Emerson, E.A. & Sistla, A.P. (1986). Automatic verification of finite-state concurrent systems. *TOPLAS*, 8(2), 244–263.
- Good, I.J. (1965). Speculations concerning the first ultraintelligent machine.